# 09 Signal Search

**Purpose.** Main evaluation: every task against every representation, with permutation controls.

**Thesis reference.** §5 Results

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [12]:
PROFILE_DIR  = "profiles"
LABELS_DIR   = "labels"
EXTRA_LABELS = []
OUT_FILE     = "signal_search.json"
PERMUTATIONS = 20
CV_REPEATS   = 5
MAX_FEATURES = 3000

ARGS = ["--profiles", PROFILE_DIR,
        "--labels-dir", LABELS_DIR,
        "--extra-labels", *EXTRA_LABELS,
        "--out", OUT_FILE,
        "--permutations", str(PERMUTATIONS),
        "--cv-repeats", str(CV_REPEATS),
        "--max-features", str(MAX_FEATURES)]

In [13]:
import argparse
import json
import warnings
from pathlib import Path

import numpy as np
from scipy.stats import entropy as shannon_entropy
from scipy.stats import kurtosis, norm, skew, spearmanr
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 42

class PLSDA(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=3):
        self.n_components = n_components

    def fit(self, X, y):
        n = min(self.n_components, X.shape[1], len(X) - 1)
        self.pls_ = PLSRegression(n_components=max(1, n))
        self.pls_.fit(X, np.asarray(y, dtype=float))
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, X):
        s = self.pls_.predict(X).ravel()
        p = 1.0 / (1.0 + np.exp(-(s - 0.5) * 6))
        return np.column_stack([1 - p, p])

class VarianceCap(BaseEstimator):
    """Keep the highest-variance features. Unsupervised: never sees y."""

    def __init__(self, k=3000):
        self.k = k

    def fit(self, X, y=None):
        if X.shape[1] <= self.k:
            self.keep_ = np.arange(X.shape[1])
        else:
            v = X.var(axis=0)
            self.keep_ = np.sort(np.argsort(v)[::-1][:self.k])
        return self

    def transform(self, X):
        return X[:, self.keep_]

def make_model(kind, k=3000):
    clf = (PLSDA(3) if kind == "plsda"
           else LogisticRegression(C=0.1, max_iter=3000,
                                   class_weight="balanced"))
    return Pipeline([("cap", VarianceCap(k)),
                     ("sc", StandardScaler()),
                     ("clf", clf)])

def cv_auc(X, y, kind, n_splits=5, n_repeats=5, seed=SEED, k=3000):
    rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats,
                                   random_state=seed)
    a = []
    for tr, te in rskf.split(X, y):
        if len(np.unique(y[te])) < 2:
            continue
        try:
            p = make_model(kind, k).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
            a.append(roc_auc_score(y[te], p))
        except Exception:
            continue
    return float(np.mean(a)) if a else 0.5

def intensity_features(F):
    """Describe the shape of each patient's intensity distribution."""
    out = []
    for row in F:
        nz = row[row > 0]
        if len(nz) < 10:
            out.append(np.zeros(18))
            continue
        tot = nz.sum()
        srt = np.sort(nz)[::-1]
        cum = np.cumsum(srt) / tot
        p = nz / tot
        n = len(nz)
        feats = [
            np.log1p(tot), np.log1p(len(nz)),
            *np.log1p(np.percentile(nz, [25, 50, 75, 90, 95, 99])),
            cum[max(0, int(0.001 * n))],      # TIC share of top 0.1%
            cum[max(0, int(0.01 * n))],       # top 1%
            cum[max(0, int(0.05 * n))],       # top 5%
            cum[max(0, int(0.10 * n))],       # top 10%
            shannon_entropy(p),
            shannon_entropy(p) / np.log(n),   # normalised entropy
            skew(np.log1p(nz)), kurtosis(np.log1p(nz)),
            np.log1p(nz.max() / max(np.median(nz), 1e-9)),
            float((np.arange(1, n + 1) * srt).sum() / (n * tot) * 2 - 1),  # Gini
        ]
        out.append(np.array(feats, dtype=float))
    X = np.vstack(out)
    return np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

def build_features(prof_dir, pids):
    prof = Path(prof_dir)
    with open(prof / "profile_pids.json") as fh:
        allp = json.load(fh)
    idx = [allp.index(p) for p in pids]

    F = np.load(prof / "mz_profiles.npy")[idx].astype(np.float64)
    feats = {"intensity_stats": intensity_features(F)}

    def tic_log(X):
        t = X.sum(axis=1, keepdims=True)
        t[t == 0] = 1
        return np.log1p(X / t * 1e6)

    feats["profile_1d"] = tic_log(F)

    cm = prof / "coarse_maps.npy"
    if cm.exists():
        C = np.load(cm)[idx].astype(np.float64)
        feats["rt_profile"] = tic_log(C.sum(axis=2))
        feats["map_2d"] = tic_log(C.reshape(len(C), -1))
    return feats

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--profiles", required=True)
    ap.add_argument("--labels-dir", default="labels")
    ap.add_argument("--extra-labels", nargs="*", default=["labels_id_matched.json"])
    ap.add_argument("--out", default="signal_search.json")
    ap.add_argument("--permutations", type=int, default=20)
    ap.add_argument("--cv-repeats", type=int, default=5)
    ap.add_argument("--max-features", type=int, default=3000)
    args = ap.parse_args()

    label_files = sorted(Path(args.labels_dir).glob("labels_*.json"))
    label_files += [Path(f) for f in args.extra_labels if Path(f).exists()]
    if not label_files:
        raise SystemExit("No label files found.")

    prof = Path(args.profiles)
    with open(prof / "profile_pids.json") as fh:
        all_pids = json.load(fh)

    print(f"Label files: {len(label_files)}")
    for f in label_files:
        print(f"  {f.name}")
    print(f"\nPermutations per comparison: {args.permutations}")
    print(f"Minimum achievable empirical p: "
          f"{1/(args.permutations+1):.3f}\n")

    rows = []
    for lf in label_files:
        with open(lf) as fh:
            labels = json.load(fh).get("labels")
        pids = [p for p in all_pids if p in labels]
        y = np.array([labels[p] for p in pids])
        if len(np.unique(y)) < 2 or min(np.bincount(y)) < 15:
            print(f"[skip] {lf.stem}: too few in one class")
            continue

        task = lf.stem.replace("labels_", "")
        ids = np.array([int("".join(c for c in p if c.isdigit())) for p in pids])
        feats = build_features(args.profiles, pids)

        print(f"\n=== {task}  n={len(y)} ({int(y.sum())}/{int((1-y).sum())}) ===")
        print(f"{'features':17s} {'model':7s} {'obs':>7} {'perm':>7} "
              f"{'gap':>8} {'z':>6} {'p(z)':>8} {'rho(ID)':>8}")
        print("-" * 76)

        for fname, X in feats.items():
            # how strongly does this representation encode acquisition order?
            pc = X - X.mean(0)
            u, s_, vt = np.linalg.svd(pc, full_matrices=False)
            rho = spearmanr(ids, u[:, 0])[0] if len(ids) > 5 else np.nan

            for kind in ("ridge", "plsda"):
                obs = cv_auc(X, y, kind, n_repeats=args.cv_repeats,
                             k=args.max_features)
                rng = np.random.default_rng(SEED)
                perms = [cv_auc(X, rng.permutation(y), kind, n_repeats=1,
                                seed=1000 + i, k=args.max_features)
                         for i in range(args.permutations)]
                pm = float(np.mean(perms))
                ps = float(np.std(perms, ddof=1)) if len(perms) > 1 else 0.0
                gap = obs - pm
                emp = (sum(p >= obs for p in perms) + 1) / (len(perms) + 1)
                # Empirical p cannot fall below 1/(N+1). The z-score against
                # the permutation distribution has no such floor and gives
                # usable resolution for multiple-comparison correction.
                z = gap / ps if ps > 1e-9 else 0.0
                zp = float(norm.sf(z)) if z > 0 else 1.0
                flag = ""
                if zp <= 0.05 and gap > 0.05:
                    flag = "  <<<"
                print(f"{fname:17s} {kind:7s} {obs:7.4f} {pm:7.4f} "
                      f"{gap:+8.4f} {z:6.2f} {zp:8.4f} {rho:+8.3f}{flag}")
                rows.append(dict(task=task, features=fname, model=kind,
                                 n=int(len(y)), observed=obs, permuted=pm,
                                 perm_std=ps, gap=gap, z=z, z_p=zp,
                                 emp_p=emp, rho_id=float(rho)))

    # summary
    print("\n" + "=" * 74)
    n_comp = len(rows)
    bonf = 0.05 / max(n_comp, 1)
    print(f"{n_comp} comparisons. Bonferroni threshold: p < {bonf:.4f}")
    print(f"Expected false positives at p<0.05: {0.05*n_comp:.1f}\n")

    rows.sort(key=lambda r: -r["gap"])
    print("Ranked by gap (observed - permuted):\n")
    print(f"{'task':26s} {'features':17s} {'model':7s} {'gap':>8} "
          f"{'p(z)':>8} {'survives Bonf':>14}")
    print("-" * 84)
    for r in rows[:12]:
        surv = "yes" if r["z_p"] < bonf else "no"
        print(f"{r['task']:26s} {r['features']:17s} {r['model']:7s} "
              f"{r['gap']:+8.4f} {r['z_p']:8.4f} {surv:>14}")

    hits = [r for r in rows if r["z_p"] <= 0.05 and r["gap"] > 0.05]
    strong = [r for r in rows if r["z_p"] < bonf and r["gap"] > 0.05]
    print(f"\n{len(hits)} comparison(s) at p < 0.05 with gap > 0.05")
    print(f"{len(strong)} surviving Bonferroni correction")

    pc = [r for r in rows if r["task"] == "positive_control_diabetes"]
    if pc:
        best = max(pc, key=lambda r: r["gap"])
        print(f"\nPOSITIVE CONTROL (diabetes): best gap {best['gap']:+.4f} "
              f"(p = {best['z_p']:.4f}) using {best['features']}")
        if best["gap"] > 0.10 and best["z_p"] <= 0.05:
            print("  -> The pipeline CAN detect a real biological signal.")
            print("     A negative CKD result is therefore about CKD, not method.")
        else:
            print("  -> Diabetes is NOT detected either. The negative CKD result")
            print("     cannot be attributed to CKD specifically --- it may be the")
            print("     data, the preprocessing, or the cohort size. Report this.")
    else:
        print("\nNo diabetes positive control found. Rerun 00b_build_labels "
              "to generate labels_positive_control_diabetes.json.")

    with open(args.out, "w") as fh:
        json.dump(rows, fh, indent=2)
    print(f"\nWritten to {args.out}")

In [14]:
import sys
sys.argv = ["10_signal_search.py"] + ARGS
main()

Label files: 15
  labels_check_diabetic.json
  labels_check_nondiabetic.json
  labels_detect_any_ckd.json
  labels_detect_mckd.json
  labels_detect_nondiabetic.json
  labels_dkd_vs_nondiabetic_mckd.json
  labels_id_matched.json
  labels_legacy_severe_only.json
  labels_main_any_ckd.json
  labels_matched_nondiabetic.json
  labels_positive_control_diabetes.json
  labels_progression.json
  labels_severity_all.json
  labels_severity_diabetic.json
  labels_severity_nondiabetic.json

Permutations per comparison: 20
Minimum achievable empirical p: 0.048


=== check_diabetic  n=191 (151/40) ===
features          model       obs    perm      gap      z     p(z)  rho(ID)
----------------------------------------------------------------------------
intensity_stats   ridge    0.6686  0.5419  +0.1268   1.81   0.0354   -0.085  <<<
intensity_stats   plsda    0.6690  0.5417  +0.1273   2.00   0.0226   -0.085  <<<
profile_1d        ridge    0.7112  0.5080  +0.2032   2.56   0.0052   -0.214  <<<
profile_1d